In [1]:
from pathlib import Path

import librosa
import soundfile as sf
import numpy as np
import pandas as pd

In [2]:
BASE_DIR = Path.cwd().parent

SELECTED_DIR = BASE_DIR / "data" / "selected"
COMMON_VOICE_DIR = BASE_DIR / "data" / "raw" / "clips"

METADATA_PATH = SELECTED_DIR / "selected_metadata.csv"

OUTPUT_DIR = BASE_DIR / "data" / "processed" / "clean_speech"
REPORT_DIR = BASE_DIR / "results" / "metrics"

TARGET_SR = 16000

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", BASE_DIR)
print("Common Voice directory:", COMMON_VOICE_DIR)
print("Metadata:", METADATA_PATH)
print("Output directory:", OUTPUT_DIR)

Project directory: d:\Kuliah\Pengolahan Sinyal Digital\PROJECT
Common Voice directory: d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\raw\clips
Metadata: d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\selected_metadata.csv
Output directory: d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\processed\clean_speech


In [3]:
print("Folder audio ditemukan:", COMMON_VOICE_DIR.exists())
print("Metadata ditemukan:", METADATA_PATH.exists())

metadata = pd.read_csv(METADATA_PATH)

print("Jumlah audio terpilih:", len(metadata))
display(metadata.head())

Folder audio ditemukan: True
Metadata ditemukan: True
Jumlah audio terpilih: 20


,client_id,path,sentence_id,sentence,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment,clip,duration[ms],duration_seconds,file_exists,selected_path
0,edb5f8670317b0c4c1e4d0321c4643bdbf207f9a8bd583...,common_voice_id_20967378.mp3,4a46323933c8b8e083ebffd84972117849e3652057bba0...,Apa yang membuat Anda memutuskan untuk bekerja...,NaN,2,0,teens,male_masculine,NaN,NaN,id,NaN,common_voice_id_20967378.mp3,5136,5.136,True,..\data\selected\cv_001.mp3
1,d8f4d9ae414f5f5c16b558e5bc8441d776ff03c66f43d3...,common_voice_id_24477232.mp3,4fc692a666fc3f60d7e4b00a10eb26212d396d29940121...,Mengapa kalian tidak mau katakan kebenaran pad...,NaN,2,0,teens,male_masculine,NaN,NaN,id,NaN,common_voice_id_24477232.mp3,5616,5.616,True,..\data\selected\cv_002.mp3
2,67b7f732e49b9008155004f31174fc4dae9fe4d02a7a94...,common_voice_id_28657647.mp3,9052f6f48deb03875c7e767c7bce22d8be25aaec0659b6...,Saya diminta untuk meneruskan pesan ini kepadamu.,NaN,2,0,NaN,NaN,NaN,NaN,id,NaN,common_voice_id_28657647.mp3,5328,5.328,True,..\data\selected\cv_003.mp3
3,fdf547996e8ecd618e46c3414bbb6870a9aa8c0c073589...,common_voice_id_35928973.mp3,99669f2438ace7bf9acbc5c73a38f78125003a5b9c9c51...,"Ketika saya datang ke sini, saya tidak kenal s...",NaN,2,0,NaN,NaN,NaN,NaN,id,NaN,common_voice_id_35928973.mp3,5220,5.220,True,..\data\selected\cv_004.mp3
4,87683f95098b70964fef02304cd68b0773e805e9fdf60d...,common_voice_id_24142794.mp3,25b246165b2882a0895c6219217caf3c7cd4dedcfe0cb2...,Sepertinya kamu akan mendapatkan perkerjaan ya...,NaN,2,1,NaN,NaN,NaN,NaN,id,NaN,common_voice_id_24142794.mp3,6516,6.516,True,..\data\selected\cv_005.mp3


In [4]:
def preprocess_audio(file_path, source_type):

    try:
        # Membaca audio tanpa mengubah sample rate
        audio, original_sr = librosa.load(
            file_path,
            sr=None,
            mono=True
        )

        if len(audio) == 0:
            raise ValueError("Audio kosong")

        if not np.all(np.isfinite(audio)):
            raise ValueError("Audio mengandung nilai tidak valid")

        # Resampling menjadi 16 kHz
        if original_sr != TARGET_SR:
            audio = librosa.resample(
                audio,
                orig_sr=original_sr,
                target_sr=TARGET_SR
            )

        peak = np.max(np.abs(audio))

        # Nama file output
        output_name = f"{source_type}_{file_path.stem}.wav"
        output_path = OUTPUT_DIR / output_name

        # Simpan sebagai WAV PCM 16-bit
        sf.write(
            output_path,
            audio,
            TARGET_SR,
            subtype="PCM_16"
        )

        return {
            "filename": output_name,
            "source_type": source_type,
            "original_filename": file_path.name,
            "original_sample_rate": original_sr,
            "processed_sample_rate": TARGET_SR,
            "duration_seconds": round(len(audio) / TARGET_SR, 3),
            "peak_amplitude": round(float(peak), 5),
            "clipping_warning": bool(peak >= 0.999),
            "output_path": str(output_path),
            "status": "SUCCESS"
        }

    except Exception as error:

        return {
            "filename": file_path.name,
            "source_type": source_type,
            "status": "FAILED",
            "error": str(error)
        }

In [5]:
results = []

for filename in metadata["path"]:

    file_path = COMMON_VOICE_DIR / filename

    if not file_path.exists():
        print("File tidak ditemukan:", filename)
        continue

    result = preprocess_audio(
        file_path,
        "common_voice"
    )

    results.append(result)

    print(filename, "->", result["status"])

common_voice_id_20967378.mp3 -> SUCCESS
common_voice_id_24477232.mp3 -> SUCCESS
common_voice_id_28657647.mp3 -> SUCCESS
common_voice_id_35928973.mp3 -> SUCCESS
common_voice_id_24142794.mp3 -> SUCCESS
common_voice_id_20899278.mp3 -> SUCCESS
common_voice_id_22146951.mp3 -> SUCCESS
common_voice_id_28161798.mp3 -> SUCCESS
common_voice_id_20338960.mp3 -> SUCCESS
common_voice_id_26709954.mp3 -> SUCCESS
common_voice_id_20387848.mp3 -> SUCCESS
common_voice_id_20952818.mp3 -> SUCCESS
common_voice_id_35113458.mp3 -> SUCCESS
common_voice_id_19192818.mp3 -> SUCCESS
common_voice_id_30272791.mp3 -> SUCCESS
common_voice_id_21553536.mp3 -> SUCCESS
common_voice_id_22708805.mp3 -> SUCCESS
common_voice_id_35281927.mp3 -> SUCCESS
common_voice_id_24069576.mp3 -> SUCCESS
common_voice_id_40088845.mp3 -> SUCCESS


In [6]:
df = pd.DataFrame(results)

manifest_path = REPORT_DIR / "clean_speech_manifest.csv"

df.to_csv(manifest_path, index=False)

print("Preprocessing selesai!")
print("Jumlah berhasil:", (df["status"] == "SUCCESS").sum())
print("Jumlah gagal:", (df["status"] == "FAILED").sum())

display(df)

Preprocessing selesai!
Jumlah berhasil: 20
Jumlah gagal: 0


,filename,source_type,original_filename,original_sample_rate,processed_sample_rate,duration_seconds,peak_amplitude,clipping_warning,output_path,status
0,common_voice_common_voice_id_20967378.wav,common_voice,common_voice_id_20967378.mp3,48000,16000,5.136,0.51674,False,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...,SUCCESS
1,common_voice_common_voice_id_24477232.wav,common_voice,common_voice_id_24477232.mp3,32000,16000,5.616,0.95134,False,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...,SUCCESS
2,common_voice_common_voice_id_28657647.wav,common_voice,common_voice_id_28657647.mp3,32000,16000,5.328,0.80500,False,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...,SUCCESS
3,common_voice_common_voice_id_35928973.wav,common_voice,common_voice_id_35928973.mp3,32000,16000,5.220,0.47584,False,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...,SUCCESS
4,common_voice_common_voice_id_24142794.wav,common_voice,common_voice_id_24142794.mp3,32000,16000,6.516,0.78769,False,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...,SUCCESS
5,common_voice_common_voice_id_20899278.wav,common_voice,common_voice_id_20899278.mp3,48000,16000,5.136,0.46299,False,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...,SUCCESS
6,common_voice_common_voice_id_22146951.wav,common_voice,common_voice_id_22146951.mp3,48000,16000,7.560,0.82824,False,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...,SUCCESS
7,common_voice_common_voice_id_28161798.wav,common_voice,common_voice_id_28161798.mp3,32000,16000,5.148,0.35809,False,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...,SUCCESS
8,common_voice_common_voice_id_20338960.wav,common_voice,common_voice_id_20338960.mp3,48000,16000,6.648,0.41686,False,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...,SUCCESS
9,common_voice_common_voice_id_26709954.wav,common_voice,common_voice_id_26709954.mp3,32000,16000,5.868,0.93959,False,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\da...,SUCCESS
